In [1]:
import pandas as pd

In [2]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')
land_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\land_df.xlsx')

In [ ]:
energy_id_to_remove = [
# less than 50% of NRJ = GHG
BC-MAIN-599152a0
ON-MAIN-1f126a43
ON-MAIN-7f050560
GRP-147b3123
ON-MAIN-7607a50e

# more than 1;5 x
NU-MAIN-8b0264c9
QC-MAIN-9de9bb0d
ON-MAIN-cb85213a
ON-MAIN-6e9be24e

]

In [4]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [5]:
from utils.data_manipulations import build_activity_name, add_site_id

In [6]:
production_df

,main_id,facility_group_id,facility_name,facility_group_name,province,facility_type,mining_processing_type,npv,archetypes,archetypes_detailed,...,Ni_conc,Mo_conc,Zn_conc,Pb_conc,Fe_conc,Pt_conc,Pd_conc,U_conc,Nb_conc,Au_conc
0,QC-MAIN-089f3c60,NaN,Bloom Lake,NaN,Quebec,mining,Open-pit,NaN,Magnetite concentrator,Only mining,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000
1,BC-MAIN-857b7b89,NaN,Brucejack,NaN,British Columbia,mining,"Underground, concentrator",Low and High Shrub Tundra,Free-milling,Might contain refractory ore?,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.920410
2,QC-MAIN-de3d8b7b,NaN,Canadian Electrolytic Zinc Limited (CEZinc),NaN,Quebec,manufacturing,Refinery,NaN,Zn refinery,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
3,QC-MAIN-e7e6a960,NaN,Canadian Malartic,NaN,Quebec,mining,"Open-pit, concentrator",Cool Mixed Forest,Free-milling,Might contain refractory ore?,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,29.039085
4,NL-MAIN-dd723db4,NaN,Carol Lake,NaN,Newfoundland and Labrador,mining,"Open-pit, concentrator",Cold Evergreen Needleleaf Forest,Fe concentrator + pellet plant,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,NaN,GRP-0a2c0d69,NaN,Meadowbank complex,Nunavut,mining,"Open-pit, underground",NaN,Free-milling,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,17.667601
62,NaN,GRP-0d911886,NaN,Porcupine complex,Ontario,mining,"Open-pit, underground",NaN,Free-milling,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,8.251949
63,NaN,GRP-147b3123,NaN,Timmins Operation,Ontario,mining,"Underground, concentrator",NaN,Free-milling,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,4.218015
64,NaN,GRP-14bfbb82,NaN,Seabee Gold Operation,Saskatchewan,mining,"Underground, concentrator",NaN,Free-milling,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.230080


In [7]:
# Add site_id to dataframes
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)
land_df = add_site_id(land_df)

In [8]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [9]:
# Add NPV to land df, and put 'Unspecified NPV' for missing values
land_df = land_df.merge(production_df[['site_id', 'npv']], on='site_id', how='left')

In [10]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
land_df = land_df.merge(production_df[['site_id', 'activity_name','archetypes']], on='site_id', how='left')

In [11]:
# To avoid double counting
biosphere_df = biosphere_df[~((biosphere_df['source_id'] == 'https://www.canada.ca/en/environment-climate-change/services/environmental-indicators/greenhouse-gas-emissions/large-facilities.html; https://open.canada.ca/data/en/dataset/a8ba14b7-7f23-462a-bdbb-83b0ef629823') & (biosphere_df['substance_id'] == 'NA - GHG'))]

In [12]:
# Assign 'substance_name' to 'substance_id' for CO2, CH4 and N2O
substance_dict = {
    '124-38-9': 'Carbon Dioxide',
    '74-82-8': 'Methane',
    '10024-97-2': 'Nitrous Oxide',
}

mask = biosphere_df['substance_id'].isin(substance_dict)
biosphere_df.loc[mask, 'substance_name'] = (
    biosphere_df.loc[mask, 'substance_id'].map(substance_dict)
)

In [13]:
substance_CO2 = ['124-38-9']
release_pathway = ['Stationary Fuel Combustion', 'On-site Transportation']
CO2_df = biosphere_df[
    biosphere_df['substance_id'].isin(substance_CO2) &
    biosphere_df['release_pathway'].isin(release_pathway)
]

# Keep only relevant columns

In [14]:
energy_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'value_MJ']
material_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'mass_t']
biosphere_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']
land_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'npv', 'area_m2', 'operation_periods']

In [15]:
energy_df = energy_df[energy_col]
material_df = material_df[material_col]
biosphere_df = biosphere_df[biosphere_col]
CO2_df = CO2_df[biosphere_col]
land_df = land_df[land_col]

In [17]:
production_df

,main_id,facility_group_id,facility_name,facility_group_name,province,facility_type,mining_processing_type,npv,archetypes,archetypes_detailed,...,Zn_conc,Pb_conc,Fe_conc,Pt_conc,Pd_conc,U_conc,Nb_conc,Au_conc,site_id,activity_name
0,QC-MAIN-089f3c60,<NA>,Bloom Lake,NaN,Quebec,mining,Open-pit,NaN,Magnetite concentrator,Only mining,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake
1,BC-MAIN-857b7b89,<NA>,Brucejack,NaN,British Columbia,mining,"Underground, concentrator",Low and High Shrub Tundra,Free-milling,Might contain refractory ore?,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.920410,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack
2,QC-MAIN-de3d8b7b,<NA>,Canadian Electrolytic Zinc Limited (CEZinc),NaN,Quebec,manufacturing,Refinery,NaN,Zn refinery,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,QC-MAIN-de3d8b7b,Refining at Canadian Electrolytic Zinc Limited...
3,QC-MAIN-e7e6a960,<NA>,Canadian Malartic,NaN,Quebec,mining,"Open-pit, concentrator",Cool Mixed Forest,Free-milling,Might contain refractory ore?,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,29.039085,QC-MAIN-e7e6a960,Open-pit mining and beneficiation at Canadian ...
4,NL-MAIN-dd723db4,<NA>,Carol Lake,NaN,Newfoundland and Labrador,mining,"Open-pit, concentrator",Cold Evergreen Needleleaf Forest,Fe concentrator + pellet plant,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.000000,NL-MAIN-dd723db4,Open-pit mining and beneficiation at Carol Lake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,<NA>,GRP-0a2c0d69,NaN,Meadowbank complex,Nunavut,mining,"Open-pit, underground",NaN,Free-milling,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,17.667601,GRP-0a2c0d69,Open-pit and underground mining at Meadowbank ...
62,<NA>,GRP-0d911886,NaN,Porcupine complex,Ontario,mining,"Open-pit, underground",NaN,Free-milling,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,8.251949,GRP-0d911886,Open-pit and underground mining at Porcupine c...
63,<NA>,GRP-147b3123,NaN,Timmins Operation,Ontario,mining,"Underground, concentrator",NaN,Free-milling,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,4.218015,GRP-147b3123,Underground mining and beneficiation at Timmin...
64,<NA>,GRP-14bfbb82,NaN,Seabee Gold Operation,Saskatchewan,mining,"Underground, concentrator",NaN,Free-milling,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,1.230080,GRP-14bfbb82,Underground mining and beneficiation at Seabee...


In [49]:
# We create a tailings_df to add the quantity of tailings as a negative material flow
tailings_df = production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'Stream', 'tailings_t_mass']]
tailings_df['flow_type'] = 'Material use'
tailings_df['subflow_type'] = 'Tailings'
tailings_df['unit'] = 't'
tailings_df['value'] = - tailings_df['tailings_t_mass']  # negative value for output in LCI
tailings_df.drop(columns=['tailings_t_mass'], inplace=True)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_9992\54366009.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['flow_type'] = 'Material use'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_9992\54366009.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = 'Tailings'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_9992\54366009.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

In [50]:
tailings_df

,site_id,activity_name,mining_processing_type,archetypes,Stream,flow_type,subflow_type,unit,value
0,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Fe concentrate,Material use,Tailings,t,-1.118700e+07
1,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings,t,-1.659991e+05
2,QC-MAIN-de3d8b7b,Refining at Canadian Electrolytic Zinc Limited...,Refinery,Zn refinery,NaN,Material use,Tailings,t,-0.000000e+00
3,QC-MAIN-e7e6a960,Open-pit mining and beneficiation at Canadian ...,"Open-pit, concentrator",Free-milling,Au-Ag dore,Material use,Tailings,t,-1.733287e+07
4,NL-MAIN-dd723db4,Open-pit mining and beneficiation at Carol Lake,"Open-pit, concentrator",Fe concentrator + pellet plant,"Fe concentrate, Fe pellets",Material use,Tailings,t,-1.647800e+07
...,...,...,...,...,...,...,...,...,...
61,GRP-0a2c0d69,Open-pit and underground mining at Meadowbank ...,"Open-pit, underground",Free-milling,Au-Ag dore,Material use,Tailings,t,-3.842631e+06
62,GRP-0d911886,Open-pit and underground mining at Porcupine c...,"Open-pit, underground",Free-milling,Au-Ag dore,Material use,Tailings,t,-2.910992e+06
63,GRP-147b3123,Underground mining and beneficiation at Timmin...,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings,t,-1.573996e+06
64,GRP-14bfbb82,Underground mining and beneficiation at Seabee...,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings,t,-1.219988e+05


In [51]:
def get_tailings_subflow_type(row):
    stream = row['Stream']

    if not isinstance(stream, str):
        return 'Tailings|Other'

    if 'Au-Ag dore' in stream:
        return 'Tailings|Gold'
    elif stream == 'Ni-Cu bulk concentrates':
        return 'Tailings|Nickel'
    elif 'Cu concentrates' or 'Cu and Mo concentrates' or 'Cu and Zn concentrates with Ag credits' in stream:
        return 'Tailings|Copper'
    else:
        return 'Tailings|Other'


In [52]:
tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_9992\4113187377.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)


In [53]:
tailings_df

,site_id,activity_name,mining_processing_type,archetypes,Stream,flow_type,subflow_type,unit,value
0,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Fe concentrate,Material use,Tailings|Copper,t,-1.118700e+07
1,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-1.659991e+05
2,QC-MAIN-de3d8b7b,Refining at Canadian Electrolytic Zinc Limited...,Refinery,Zn refinery,NaN,Material use,Tailings|Other,t,-0.000000e+00
3,QC-MAIN-e7e6a960,Open-pit mining and beneficiation at Canadian ...,"Open-pit, concentrator",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-1.733287e+07
4,NL-MAIN-dd723db4,Open-pit mining and beneficiation at Carol Lake,"Open-pit, concentrator",Fe concentrator + pellet plant,"Fe concentrate, Fe pellets",Material use,Tailings|Copper,t,-1.647800e+07
...,...,...,...,...,...,...,...,...,...
61,GRP-0a2c0d69,Open-pit and underground mining at Meadowbank ...,"Open-pit, underground",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-3.842631e+06
62,GRP-0d911886,Open-pit and underground mining at Porcupine c...,"Open-pit, underground",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-2.910992e+06
63,GRP-147b3123,Underground mining and beneficiation at Timmin...,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-1.573996e+06
64,GRP-14bfbb82,Underground mining and beneficiation at Seabee...,"Underground, concentrator",Free-milling,Au-Ag dore,Material use,Tailings|Gold,t,-1.219988e+05


In [114]:
# Maybe need to differentiate unit and value in energy_df and material_df ?
energy_df['unit'] = 'MJ'
material_df['unit'] = 't'
#tailings_df['unit'] = 't'
land_df['unit'] = 'm2'
energy_df.rename(columns={'value_MJ': 'value'}, inplace=True)
material_df.rename(columns={'mass_t': 'value'}, inplace=True)
land_df.rename(columns={'area_m2': 'value'}, inplace=True)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\3305228502.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  land_df['unit'] = 'm2'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\3305228502.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  land_df.rename(columns={'area_m2': 'value'}, inplace=True)


In [115]:
# To know the % inferred vs 'primary'
energy_df['data_source'] = 'MetalliCan'
material_df['data_source'] = 'MetalliCan'
biosphere_df['data_source'] = 'MetalliCan'
CO2_df['data_source'] = 'MetalliCan'
land_df['data_source'] = 'MetalliCan'

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\1843239807.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  biosphere_df['data_source'] = 'MetalliCan'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\1843239807.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  land_df['data_source'] = 'MetalliCan'


In [116]:
# To know the % inferred vs 'primary'
energy_df['value_formula'] = 'MetalliCan'
material_df['value_formula'] = 'MetalliCan'
biosphere_df['value_formula'] = 'MetalliCan'
CO2_df['value_formula'] = 'MetalliCan'
land_df['value_formula'] = 'MetalliCan'

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\2105971740.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  biosphere_df['value_formula'] = 'MetalliCan'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_12092\2105971740.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  land_df['value_formula'] = 'MetalliCan'


In [117]:
land_df

,site_id,activity_name,mining_processing_type,archetypes,npv,value,operation_periods,unit,data_source,value_formula
0,BC-MAIN-23155c25,NaN,Underground,NaN,NaN,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,MetalliCan
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,NaN,1.396089e+06,NaN,m2,MetalliCan,MetalliCan
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,7.967835e+06,NaN,m2,MetalliCan,MetalliCan
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,4.167369e+05,NaN,m2,MetalliCan,MetalliCan
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cool Evergreen Needleleaf Forest,1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan
...,...,...,...,...,...,...,...,...,...,...
125,GRP-7a9ba115,NaN,Electric arc furnace,NaN,NaN,1.991940e+06,NaN,m2,MetalliCan,MetalliCan
126,GRP-a13779f8,Underground mining and beneficiation at Snow Lake,"Concentrator, Underground",Free-milling,NaN,3.104131e+06,"2013–open, 1949–2000; 2021–open",m2,MetalliCan,MetalliCan
127,GRP-b48ec279,NaN,Smelter,NaN,NaN,6.630997e+06,NaN,m2,MetalliCan,MetalliCan
128,GRP-dc07540b,NaN,"Underground, Smelter, refinery, plant, Concent...",NaN,NaN,3.338002e+07,2014–open,m2,MetalliCan,MetalliCan


# Data-gap filling

In [118]:
from core.data_gap_filling import *

In [119]:
# Initialize the InferenceEngine Class
engine = InferenceEngine(
    site_df=production_df,
    production_df=production_df,
    energy_df=energy_df,
    co2_df=CO2_df,
    land_df=land_df,
    material_df=material_df,
)

## Energy

In [120]:
site_id_to_fill_nrj = production_df[production_df['infer_energy_data'] == 'Yes']['site_id'].tolist()

In [121]:
ef = {
    "diesel": 2681,
    "natural_gas": 2354,
    "lpg": 2753
}

stationary_share_rules = {
    "Open-pit, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
    "Underground, concentrator": {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1},
}

default_shares = {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1}

In [122]:
combined_energy_df, inferred_energy_df = engine.infer_energy_for_sites(
    site_ids=site_id_to_fill_nrj,
    ef_co2_per_unit=ef,
    stationary_share_rules=stationary_share_rules,
    default_shares=default_shares
)


In [123]:
inferred_energy_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula
0,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Diesel|Transport,6.613078e+05,L,Inference from CO2 (transport diesel),(1772.9661 * 1e6) / 2681
1,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Diesel|Stationary,2.361087e+07,L,Inference from CO2 (stationary diesel),(63300.737969999995 * 1e6) / 2681
2,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Natural Gas|Stationary,5.378143e+07,m3,Inference from CO2 (stationary natural_gas),(126601.47593999999 * 1e6) / 2354
3,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,Energy,Lpg|Stationary,7.664455e+06,L,Inference from CO2 (stationary lpg),(21100.24599 * 1e6) / 2753
4,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Diesel|Transport,8.863841e+05,L,Inference from CO2 (transport diesel),(2376.3958 * 1e6) / 2681
5,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Diesel|Stationary,3.291620e+07,L,Inference from CO2 (stationary diesel),(88248.34095 * 1e6) / 2681
6,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Natural Gas|Stationary,7.497735e+07,m3,Inference from CO2 (stationary natural_gas),(176496.6819 * 1e6) / 2354
7,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,Energy,Lpg|Stationary,1.068511e+07,L,Inference from CO2 (stationary lpg),(29416.113650000003 * 1e6) / 2753
8,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Energy,Diesel|Transport,1.738740e+07,L,Inference from CO2 (transport diesel),(46615.6208 * 1e6) / 2681
9,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Energy,Diesel|Stationary,2.394360e+05,L,Inference from CO2 (stationary diesel),(641.9279999999999 * 1e6) / 2681


## Material

In [124]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [125]:
material_archetype_rules_df = pd.read_excel(r'data/SI/SI_LCIs_review.xlsx', sheet_name='Mat_inference')

In [126]:
engine.init_material_inference(material_archetype_rules_df)

In [127]:
combined_material_df, inferred_material_df = engine.infer_material_for_sites(
    site_ids=site_id_to_fill_material,
    overwrite=False,
)

⚠️ No material rules for archetype 'Co hydrometallurgical refinery'
⚠️ No material rules for archetype 'Zn refinery'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Fe concentrator'
⚠️ No material rules for archetype 'Ni hydrometallurgical refinery'
⚠️ No material rules for archetype 'Fe concentrator + pellet plant'
⚠️ No material rules for archetype 'Direct Shipping Ore'
⚠️ No material rules for archetype 'Ni-Cu smelter'
⚠️ No material rules for archetype 'Cu-Ni smelter and refinery'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'Cu smelter'
⚠️ No material rules for archetype 'Nb smelter'
⚠️ No material rules for archetype 'Nb mine'
⚠️ No material rules for archetype 'Zn refinery'
⚠️ No material rules for archetype 'Hybrid free-milling-refactory'
⚠️ No material rules for archetype 'High-grade U acid-leach mill

C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.material_df, inferred], ignore_index=True)


In [128]:
inferred_material_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Material,Anti-scalant,None,kg,Archetype inference (nan) | MetalliCan,0.012 * ore_processed_t,None,None,NaN,NaN,NaN
1,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Material,Floculant,None,kg,Archetype inference (nan) | MetalliCan,0.063 * ore_processed_t,None,None,NaN,NaN,NaN
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Material,Frother,None,kg,Archetype inference (nan) | MetalliCan,0.018 * ore_processed_t,None,None,NaN,NaN,NaN
3,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Material,Grinding media,None,kg,Archetype inference (nan) | MetalliCan,0.189 * ore_processed_t,None,None,NaN,NaN,NaN
4,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Material,Lime,None,kg,Archetype inference (nan) | MetalliCan,0.889 * ore_processed_t,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,GRP-a13779f8,Underground mining and beneficiation at Snow Lake,"Underground, concentrator",Free-milling,Material,Grinding media,None,kg,Archetype inference (Comminution) | Norgate & ...,0.71 * ore_processed_t,None,None,NaN,NaN,NaN
167,GRP-a13779f8,Underground mining and beneficiation at Snow Lake,"Underground, concentrator",Free-milling,Material,Lime,None,kg,Archetype inference (Extraction and recovery -...,2.2 * ore_processed_t,None,None,NaN,NaN,NaN
168,GRP-a13779f8,Underground mining and beneficiation at Snow Lake,"Underground, concentrator",Free-milling,Material,Sodium cyanide,None,kg,Archetype inference (Extraction and recovery -...,0.64 * ore_processed_t,None,None,NaN,NaN,NaN
169,GRP-a13779f8,Underground mining and beneficiation at Snow Lake,"Underground, concentrator",Free-milling,Material,Activated carbon,None,g,Archetype inference (Extraction and recovery -...,24 * ore_processed_t,None,None,NaN,NaN,NaN


## Cement

In [129]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [130]:
cement_params = {
    "underground": {
        "cement_factor": (5, 50),
        "backfill_share": (0.3, 0.9),  # optional, future
    },
    "open_pit": None,
}


In [131]:
combined_cement_df, inferred_cement_df = engine.infer_cement_for_sites(
    site_ids=site_id_to_fill_material,
    cement_params=cement_params,
)

C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:137: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.material_df = pd.concat(


In [132]:
inferred_cement_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",8.300000e+05,4.565000e+06,8.300000e+06
1,MB-MAIN-e0a6250e,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.323529e+06,1.277941e+07,2.323529e+07
2,NL-MAIN-2d8801d6,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",3.970588e+06,2.183824e+07,3.970588e+07
3,NU-MAIN-8b0264c9,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",9.590715e+06,5.274893e+07,9.590715e+07
4,ON-MAIN-1f126a43,Underground mining and beneficiation at Macassa,"Underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.207940e+06,1.214367e+07,2.207940e+07
5,ON-MAIN-206041d1,Underground mining at Fraser,Underground,"Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.765165e+06,1.520841e+07,2.765165e+07
6,ON-MAIN-28f3f0fc,Underground mining at Totten,Underground,"Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.070000e+06,1.138500e+07,2.070000e+07
7,ON-MAIN-48fe2205,Underground mining at Garson,Underground,"Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",3.080000e+06,1.694000e+07,3.080000e+07
8,ON-MAIN-4e0734b5,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.285000e+06,1.256750e+07,2.285000e+07
9,ON-MAIN-52224e1e,Underground mining at Creighton,Underground,"Porphyry sulfide, flotation-based",Material,Cement (backfilling),None,kg,Inference | Cement backfilling | underground |...,"ore_processed_t * Uniform(5, 50)",ore_processed_t,"Uniform(5, 50)",2.165000e+06,1.190750e+07,2.165000e+07


## Explosives

In [133]:
site_id_to_fill_explosives = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [134]:
explosives_params = {
    "open_pit": {
        "strip_ratio": (1.5, 6.0),       # t waste / t ore
        "explosive_factor": (0.25, 0.8), # kg explosives / t material
    },
    "underground": {
        "explosive_factor": (0.1, 0.3),  # kg explosives / t ore
    }
}

In [135]:
combined_explosives_df, inferred_explosives_df = engine.infer_explosives_for_sites(
    site_ids=site_id_to_fill_explosives,
    explosive_params=explosives_params,
)

C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.material_df, inferred], ignore_index=True)


## Land

In [136]:
site_id_to_fill_land = production_df[production_df['infer_land_data'] == 'Yes']['site_id'].tolist()

In [137]:
combined_land_df, inferred_land_df = engine.infer_land_for_sites(
    site_ids=site_id_to_fill_land,
    formula_open_pit="0.791 * ore_processed_t - 7.76e5",
    formula_underground="Uniform(5e5, 2e6)",
    formula_other="Uniform(1e4, 1e5)",
    overwrite=False
)


C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\core\data_gap_filling.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([self.land_df, inferred], ignore_index=True)


In [138]:
inferred_land_df

,site_id,activity_name,mining_processing_type,archetypes,value,unit,operation_periods,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,None,m2,None,Inference (other facilities range),"Uniform(1e4, 1e5)",None,"Uniform(1e4, 1e5)",1.000000e+04,5.500000e+04,1.000000e+05
1,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,None,m2,None,Inference (other facilities range),"Uniform(1e4, 1e5)",None,"Uniform(1e4, 1e5)",1.000000e+04,5.500000e+04,1.000000e+05
2,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Hybrid free-milling-refactory,None,m2,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",5.000000e+05,1.250000e+06,2.000000e+06
3,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,None,m2,None,Inference (open-pit model),(0.510 * 5600000.0) - 7.14e6,ore_processed_t,None,3.653600e+06,3.653600e+06,3.653600e+06
4,NU-MAIN-8b0264c9,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,None,m2,None,Inference (open-pit model),(0.510 * 1918143.0) - 7.14e6,ore_processed_t,None,7.412511e+05,7.412511e+05,7.412511e+05
5,ON-MAIN-687b8c8d,Underground mining and beneficiation at Island,"Underground, concentrator",Free-milling,None,m2,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",5.000000e+05,1.250000e+06,2.000000e+06
6,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,None,m2,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",5.000000e+05,1.250000e+06,2.000000e+06
7,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",None,m2,None,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,8.014118e+04,8.014118e+04,8.014118e+04
8,QC-MAIN-649d2873,Converting at Niobec Converter,Convertor,Nb smelter,None,m2,None,Inference (other facilities range),"Uniform(1e4, 1e5)",None,"Uniform(1e4, 1e5)",1.000000e+04,5.500000e+04,1.000000e+05


In [139]:
combined_land_df['npv'] = combined_land_df['npv'].fillna('Unspecified NPV')

In [140]:
combined_land_df

,site_id,activity_name,mining_processing_type,archetypes,npv,value,operation_periods,unit,data_source,value_formula,amount_parameter,parameter_distribution,value_min,value_mean,value_max
0,BC-MAIN-23155c25,NaN,Underground,NaN,Unspecified NPV,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,Unspecified NPV,1.396089e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,7.967835e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,4.167369e+05,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cool Evergreen Needleleaf Forest,1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,NU-MAIN-8b0264c9,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Free-milling,Unspecified NPV,NaN,None,m2,Inference (open-pit model),(0.510 * 1918143.0) - 7.14e6,ore_processed_t,None,741251.113000,7.412511e+05,7.412511e+05
135,ON-MAIN-687b8c8d,Underground mining and beneficiation at Island,"Underground, concentrator",Free-milling,Unspecified NPV,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
136,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Unspecified NPV,NaN,None,m2,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",500000.000000,1.250000e+06,2.000000e+06
137,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Unspecified NPV,NaN,None,m2,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,80141.176471,8.014118e+04,8.014118e+04


# Normalize flows

In [142]:
combined_material_df = pd.concat([combined_material_df, combined_cement_df, combined_explosives_df, tailings_df], ignore_index=True)

In [143]:
from core.normalization_allocation import normalize_flows, normalize_land_flows

### Per ore processed

In [144]:
energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value')
material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='value')
biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

### Per concentrate stream

In [47]:
energy_conc_econ = normalize_flows(combined_energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [48]:
material_conc_econ = normalize_flows(combined_material_df, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='value')

In [49]:
biosphere_conc_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [50]:
land_conc_econ = normalize_land_flows(combined_land_df, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [51]:
land_conc_econ.loc[
    land_conc_econ["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [52]:
land_conc_econ

,site_id,flow_type,functional_unit,normalization_key,allocation_factor,unit,activity_name,mining_processing_type,archetypes,npv,operation_periods,data_source,value_formula,amount_parameter,parameter_distribution,value_normalized,value_min_normalized,value_mean_normalized,value_max_normalized,reference_mass_unit
0,BC-MAIN-3f490561,transformation_from,Cu concentrate,land_concentrate_economic,1.0,m2,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.008741,NaN,NaN,NaN,kg
1,BC-MAIN-3f490561,transformation_to,Cu concentrate,land_concentrate_economic,1.0,m2,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.008741,NaN,NaN,NaN,kg
2,BC-MAIN-3f490561,occupation,Cu concentrate,land_concentrate_economic,1.0,m2*year,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.174816,NaN,NaN,NaN,kg
3,BC-MAIN-4724f4ba,transformation_from,Doré,land_concentrate_economic,1.0,m2,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,167.650962,NaN,NaN,NaN,kg
4,BC-MAIN-4724f4ba,transformation_to,Doré,land_concentrate_economic,1.0,m2,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,167.650962,NaN,NaN,NaN,kg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,ON-MAIN-f4fc3276,transformation_to,Doré,land_concentrate_economic,1.0,m2,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Unspecified NPV,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",NaN,20.375403,50.938507,81.501612,kg
176,ON-MAIN-f4fc3276,occupation,Doré,land_concentrate_economic,1.0,m2*year,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Unspecified NPV,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",NaN,407.508059,1018.770148,1630.032238,kg
177,QC-MAIN-5ce331b8,transformation_from,Ni concentrate,land_concentrate_economic,1.0,m2,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Unspecified NPV,None,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,NaN,0.000011,0.000011,0.000011,kg
178,QC-MAIN-5ce331b8,transformation_to,Ni concentrate,land_concentrate_economic,1.0,m2,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Unspecified NPV,None,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,NaN,0.000011,0.000011,0.000011,kg


# Redesign land tables

# Keeping only relevant columns

In [54]:
energy_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
material_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']
biosphere_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']
land_col = ['activity_name', 'functional_unit', 'flow_type', 'npv', 'operation_periods', 'site_id', 'substance_name', 'unit', 'value_normalized', 'value_min_normalized', 'value_mean_normalized', 'value_max_normalized']

# Exports normalized dataframes

In [145]:
energy_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/energy_df.csv', index=False)
material_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/material_df.csv', index=False)
biosphere_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/biosphere_df.csv', index=False)

In [56]:
energy_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv', index=False)
material_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv', index=False)
biosphere_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv', index=False)
land_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv', index=False)

In [57]:
land_conc_econ

,site_id,flow_type,functional_unit,normalization_key,allocation_factor,unit,activity_name,mining_processing_type,archetypes,npv,operation_periods,data_source,value_formula,amount_parameter,parameter_distribution,value_normalized,value_min_normalized,value_mean_normalized,value_max_normalized,reference_mass_unit
0,BC-MAIN-3f490561,transformation_from,Cu concentrate,land_concentrate_economic,1.0,m2,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.008741,NaN,NaN,NaN,kg
1,BC-MAIN-3f490561,transformation_to,Cu concentrate,land_concentrate_economic,1.0,m2,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.008741,NaN,NaN,NaN,kg
2,BC-MAIN-3f490561,occupation,Cu concentrate,land_concentrate_economic,1.0,m2*year,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator","Porphyry sulfide, flotation-based",Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,0.174816,NaN,NaN,NaN,kg
3,BC-MAIN-4724f4ba,transformation_from,Doré,land_concentrate_economic,1.0,m2,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,167.650962,NaN,NaN,NaN,kg
4,BC-MAIN-4724f4ba,transformation_to,Doré,land_concentrate_economic,1.0,m2,Open-pit mining at Elk,Open-pit,Free-milling,Cold Evergreen Needleleaf Forest,NaN,MetalliCan,MetalliCan,NaN,NaN,167.650962,NaN,NaN,NaN,kg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,ON-MAIN-f4fc3276,transformation_to,Doré,land_concentrate_economic,1.0,m2,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Unspecified NPV,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",NaN,20.375403,50.938507,81.501612,kg
176,ON-MAIN-f4fc3276,occupation,Doré,land_concentrate_economic,1.0,m2*year,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Free-milling,Unspecified NPV,None,Inference (underground range),"Uniform(5e5, 2e6)",None,"Uniform(5e5, 2e6)",NaN,407.508059,1018.770148,1630.032238,kg
177,QC-MAIN-5ce331b8,transformation_from,Ni concentrate,land_concentrate_economic,1.0,m2,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Unspecified NPV,None,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,NaN,0.000011,0.000011,0.000011,kg
178,QC-MAIN-5ce331b8,transformation_to,Ni concentrate,land_concentrate_economic,1.0,m2,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator","Porphyry sulfide, flotation-based",Unspecified NPV,None,Inference (open-pit model),(0.510 * 1082352.94117647) - 7.14e6,ore_processed_t,None,NaN,0.000011,0.000011,0.000011,kg
